In [ ]:
import sys
import pandas as pd
import biathlonresults as br

print(sys.executable)

In [ ]:
#TESTING THE PACKAGE

race = br.results("BT2526SWRLCP01SMSP")
df = pd.DataFrame(race["Results"])
df.shape

In [ ]:
#LOOKING DEEPER INTO IT..
df.columns

In [ ]:
#looking more..
df["IRM"].value_counts(dropna=False)

In [ ]:
#Analying the way the data is accessible, looking for rangetime in 25/26 season
rng = pd.DataFrame(br.analytic_results("BT2526SWRLCP01SMSP", "RNGT")["Results"])
rng[["Rank", "Name", "Nat", "Shootings", "TotalTime"]].head(10)

In [ ]:
#looking for rangetime in old files, season 2010/2011
old = pd.DataFrame(br.analytic_results("BT1011SWRLCP01SMSP", "RNGT")["Results"])
old.shape

In [ ]:
#analyzing the shape of season 2010/11
pd.DataFrame(br.results("BT1011SWRLCP01SMSP")["Results"]).shape

In [ ]:
#Trying to filter out which season has rangetime

seasons = ["0708","0809","0910","1011", "1112", "1213", "1314", "1415", "1516", "1617", "1718",
           "1819", "1920", "2021", "2122", "2223", "2324", "2425", "2526"]

for s in seasons:
    race_id = f"BT{s}SWRLCP01SMSP"
    n_res = len(br.results(race_id)["Results"])
    n_crs = len(br.analytic_results(race_id, "CRST")["Results"])
    n_rng = len(br.analytic_results(race_id, "RNGT")["Results"])
    n_stt = len(br.analytic_results(race_id, "STTM")["Results"])
    print(s, n_res, n_crs, n_rng, n_stt)

In [ ]:
#looking deeper into one specific season

events = pd.DataFrame(br.events("2526", "1"))
events[["EventId", "ShortDescription", "StartDate"]]

Above we notice that Olympic events have another ID that finishes with "__"
We wonder if we should Olympic games and World Championships in the training data, or use it as a test data. 

In [ ]:
#looking at what each event contains
comps = pd.DataFrame(br.competitions("BT2526SWRLCP01"))
comps[["RaceId", "ShortDescription", "DisciplineId", "catId"]]

# Result: in 'DisciplineId' the attribute 'SP' corresponds to what we want

In [ ]:
#fetching the data, version 1

seasons = ["1213", "1314", "1415", "1516", "1617", "1718", "1819",
           "1920", "2021", "2122", "2223", "2324", "2425", "2526"]

rows = []
for s in seasons:
    for ev in br.events(s, "1"):
        for c in br.competitions(ev["EventId"]):
            if c["DisciplineId"] == "SP":
                rows.append({
                    "season": s,
                    "place": ev["ShortDescription"],
                    "race_id": c["RaceId"],
                    "gender": c["catId"],
                    "start": c["StartTime"],
                    "status": c["StatusText"],
                })

races = pd.DataFrame(rows)
races.shape


In [ ]:
#looking at what we got..
print(races.head())
print(races["gender"].value_counts())
print(races["status"].value_counts())


In [ ]:
#filtering out cancelled races
races[races["status"] == "Cancelled"]

In [ ]:
#keeping only the races that took place
races = races[races["status"] == "Final"].reset_index(drop=True)
races.shape

In [ ]:
# A small function that fetches 1 race and attaches the split times to each athlete.

def get_race(race_id):
    res = pd.DataFrame(br.results(race_id)["Results"])
    res = res[["IBUId", "Name", "Nat", "Bib", "Rank", "IRM",
               "Shootings", "ShootingTotal", "TotalTime", "Behind"]]
    for code in ["CRST", "RNGT", "STTM", "SKIT"]:
        ana = pd.DataFrame(br.analytic_results(race_id, code)["Results"])
        if len(ana) > 0:
            ana = ana[["IBUId", "TotalTime"]].drop_duplicates("IBUId")
            res = res.merge(ana.rename(columns={"TotalTime": code}),
                            on="IBUId", how="left")
        else:
            res[code] = None
    res["race_id"] = race_id
    return res

test = get_race("BT2526SWRLCP01SMSP")
test.head()

**Limitation:** the API's ski time field (SKIT) is empty for sprints, so penalty loop time is not available directly. Plan: derive it as total time minus course time minus range time.

**To check:** raw checkpoint passage times (intermediates) are not in the API endpoints we use. The website's live view shows them and races carry a `HasLiveData` flag, so a feed exists somewhere. With passage times we could measure the segment from range exit to the end of the penalty loop directly.

**Workaround for now:** total time = lap 1 + range 1 + X1 + lap 2 + range 2 + X2 + lap 3. We know X1 + X2 as total minus course time minus range time. On Östersund 2025 this leftover is about 10 to 12 s with clean shooting and about 22 s more per miss.

In [ ]:
# Defining the function we will use

def get_race(race_id):
    res = pd.DataFrame(br.results(race_id)["Results"])
    res = res[["IBUId", "Name", "Nat", "Bib", "Rank", "IRM",
               "Shootings", "ShootingTotal", "TotalTime", "Behind"]]
    for code in ["CRST", "RNGT", "STTM", "CRS1", "CRS2", "CRS3", "RNG1", "RNG2"]:
        ana = pd.DataFrame(br.analytic_results(race_id, code)["Results"])
        if len(ana) > 0:
            ana = ana[["IBUId", "TotalTime"]].drop_duplicates("IBUId")
            res = res.merge(ana.rename(columns={"TotalTime": code}),
                            on="IBUId", how="left")
        else:
            res[code] = None
    res["race_id"] = race_id
    return res

In [ ]:
# Downloading the data

import time

frames = []
failed = []
for i, rid in enumerate(races["race_id"]):
    try:
        frames.append(get_race(rid))
    except Exception as e:
        failed.append(rid)
        print("failed:", rid, e)
    if i % 25 == 0:
        print(i, "of", len(races))
    time.sleep(0.2)

raw = pd.concat(frames, ignore_index=True)
raw = raw.merge(races, on="race_id", how="left")

print(raw.shape, "failed:", len(failed))

Note on the data, 0 failed races. 25507 rows for 24 columns, 252 races, about 101 athletes per race

In [ ]:
# Saving the downloaded data in a .csv file
import os
os.makedirs("data", exist_ok=True)
raw.to_csv("data/sprints_raw.csv", index=False)


In [ ]:
#Check for DNF

print(raw["IRM"].value_counts(dropna=False))

cols = ["CRST", "RNGT", "STTM", "CRS1", "CRS2", "CRS3", "RNG1", "RNG2"]
print(raw[cols].replace("", pd.NA).isna().mean().round(3))

**SUMMARY**

**Finishers:** 24,957 rows, or 97.8 %. Both NaN and None mean "no special status", so they count together (2 spellings of "empty" that we'll merge in step 2).
**DNS:** 382 (1.5 %). DSQ: 89. DNF: only 79 (0.3 %).
**Split times:** missing in just 1.5 to 1.8 % of rows. CRS1 is missing for exactly the DNS share, and CRST for DNS plus DNF. So practically every finisher has complete split data.

**OPEN QUESTION WITH THE CLASS**
What to do about DNF, DNS, DSQ??

**OWN PROPOSITION**
Question 1: do we try to predict this result?
No. If an athlete has DNS, DNF or DSQ in a race, that row is removed from the races we predict. There is no valid finish position, so there is nothing to predict and nothing to check the prediction against.

Question 2: do we remember that it happened?
Yes. When we later predict that same athlete's next race, "had a DNS, DNF or DSQ recently" is information we can use.

An example: an athlete has a DNS in Östersund, then races in Hochfilzen a week later.

The Östersund row: removed.
The Hochfilzen row: kept, and it can carry a note saying "did not start last race".

And yes, I'd treat DSQ the same way, so all 3 follow 1 rule.

**Decision:** rows with DNS, DNF or DSQ are removed as prediction targets, because they have no valid finish position. They are kept as information for the athlete's later races (e.g. "did not start last race"). Together they are 550 of 25,507 rows (2.2 %).

**Limitation:** some DNF happen on a bad day, so removing them drops a few poor performances. With 79 DNF (0.3 %) the effect is small.